## 07 — Transformer Model for BFRB Detection (Milestone 2)

This notebook implements a Transformer-based sequence classifier for BFRB detection.

The model treats each gesture sequence as a token sequence (produced by `06_NLPDataPrep.ipynb`). It includes discrete symbolic tokens, self attention and token embeddings.

### Inputs (from 06_NLPDataPrep.ipynb)
- `../data/processed/train_text_seq.pkl`
- `../data/processed/test_text_seq.pkl`
- `../data/processed/nlp_vocab.pkl`
- `../data/processed/nlp_max_seq_len.pkl`

### Outputs
- `../results/transformer_results.pkl` — binary F1, macro F1, predictions

## 1. Imports

This section initializes the computational graph, core utilities and imports the specialized PyTorch components required for Transformer architectures.

In [ ]:
import pickle
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PROCESSED = Path("../data/processed")
RESULTS_DIR    = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print("Imports done.")

Device: cpu
Imports done.


## 2. Load Preprocessed Data

Load the serialized pickle files generated during `06_nlp_datapreprocessing.ipynb`.We load the discrete gesture tokens and vocabulary mapping generated in the NLP Data Prep phase. This converts sensor data into a format the Transformer can "read" as a language.

In [ ]:
with open(DATA_PROCESSED / "train_text_seq.pkl", "rb") as f:
    train_data = pickle.load(f)

with open(DATA_PROCESSED / "test_text_seq.pkl", "rb") as f:
    test_data = pickle.load(f)

with open(DATA_PROCESSED / "nlp_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open(DATA_PROCESSED / "nlp_max_seq_len.pkl", "rb") as f:
    max_seq_len = pickle.load(f)

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab["<PAD>"]

print(f"Train sequences : {len(train_data)}")
print(f"Test sequences  : {len(test_data)}")
print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Max seq length  : {max_seq_len}")
print(f"PAD index       : {PAD_IDX}")

Train sequences : 6515
Test sequences  : 1629
Vocabulary size : 38
Max seq length  : 8400
PAD index       : 0


## 3. Hyperparameters

Sequence truncation are done to keep training feasible. The model architecture are structured with embedding dimension, number of attention heads, number of transformer encoder layers, and feedforward hidden dimension inside Transformer. For the training, batch size wsas set to 32 sequences at one time, with 10 training sets and learning rate 0.001. Overall, this cell sets the size, depth and training behaviour of our custom Transformer model.

In [ ]:
# Sequence truncation
MAX_LEN     = 512   

# Model architecture
EMBED_DIM   = 64    
NUM_HEADS   = 4     # Number of attention heads (must divide EMBED_DIM)
NUM_LAYERS  = 2     # Number of Transformer encoder layers
FF_DIM      = 128   # Feed-forward hidden dimension inside Transformer
DROPOUT     = 0.1

# Training
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 0.001

print("Hyperparameters set.")
print(f"  Truncation length : {MAX_LEN}")
print(f"  Embedding dim     : {EMBED_DIM}")
print(f"  Attention heads   : {NUM_HEADS}")
print(f"  Encoder layers    : {NUM_LAYERS}")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Epochs            : {EPOCHS}")

Hyperparameters set.
  Truncation length : 512
  Embedding dim     : 64
  Attention heads   : 4
  Encoder layers    : 2
  Batch size        : 32
  Epochs            : 10


## 4. Dataset and DataLoader

This section implements a custom Dataset class that truncates sequences to our MAX_LEN and a collation function that handles variable-length padding.

In [ ]:
class BFRBTokenDataset(Dataset):
    def __init__(self, records, max_len):
        self.records = records
        self.max_len = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record    = self.records[idx]
        token_ids = record["token_ids"][:self.max_len]  # Truncate
        label     = record["label"]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)


def collate_fn(batch):
    """
    Pads sequences in a batch to the same length using PAD_IDX.
    Returns padded token IDs, padding mask, and labels.
    """
    sequences, labels = zip(*batch)
    padded    = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    # key_padding_mask: True where tokens are padding (to be ignored by attention)
    pad_mask  = (padded == PAD_IDX)
    labels    = torch.stack(labels)
    return padded, pad_mask, labels


train_dataset = BFRBTokenDataset(train_data, MAX_LEN)
test_dataset  = BFRBTokenDataset(test_data,  MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 204
Test batches  : 51


## 5. Positional Encoding

Transformers have no built-in notion of order, so we inject positional
information using sinusoidal positional encodings (Vaswani et al., 2017).
This allows the model to distinguish `acc_x_low` at timestep 1 from
`acc_x_low` at timestep 100.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding as described in:
    Vaswani et al. 'Attention is All You Need', NeurIPS 2017.
    """
    def __init__(self, embed_dim, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe     = torch.zeros(max_len, embed_dim)
        pos    = torch.arange(0, max_len).unsqueeze(1).float()
        div    = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)   # (1, max_len, embed_dim)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


print("PositionalEncoding defined.")

PositionalEncoding defined.


## 6. Transformer Classifier

Architecture:
1. **Token Embedding** — maps each token ID to a dense vector
2. **Positional Encoding** — injects sequence position information
3. **Transformer Encoder** — NUM_LAYERS of multi-head self-attention + feed-forward
4. **Mean Pooling** — averages token representations (ignoring padding)
5. **Classifier Head** — linear layer → binary prediction (BFRB vs non-BFRB)

In [ ]:
class TransformerBFRBClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers,
                 ff_dim, max_len, num_classes=2, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx   = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_enc   = PositionalEncoding(embed_dim, max_len, dropout)

        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier  = nn.Linear(embed_dim, num_classes)

    def forward(self, x, pad_mask):
        # x: (batch, seq_len)
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        emb = self.pos_enc(emb)           # add positional encoding
        enc = self.transformer(emb, src_key_padding_mask=pad_mask)  # (batch, seq_len, embed_dim)

        # Mean pooling over non-padding tokens
        mask_expanded = (~pad_mask).unsqueeze(-1).float()  # (batch, seq_len, 1)
        summed  = (enc * mask_expanded).sum(dim=1)         # (batch, embed_dim)
        lengths = mask_expanded.sum(dim=1).clamp(min=1)    # (batch, 1)
        pooled  = summed / lengths                         # (batch, embed_dim)

        return self.classifier(pooled)    # (batch, num_classes)


model = TransformerBFRBClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    max_len=MAX_LEN,
    num_classes=2,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
print(model)

Model parameters: 69,506
TransformerBFRBClassifier(
  (embedding): Embedding(38, 64, padding_idx=0)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


## 7. Training

Executes the training loop, calculating Cross-Entropy loss and updating model weights via the Adam optimizer.Train the model over multiple epochs, tracking loss and accuracy for each iteration through the dataset.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for token_ids, pad_mask, labels in loader:
        token_ids = token_ids.to(DEVICE)
        pad_mask  = pad_mask.to(DEVICE)
        labels    = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(token_ids, pad_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


print("Starting training...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

print("Training complete.")

Starting training...
Epoch 01/10 | Loss: 0.4677 | Train Acc: 0.7807
Epoch 02/10 | Loss: 0.3847 | Train Acc: 0.8249
Epoch 03/10 | Loss: 0.3534 | Train Acc: 0.8381
Epoch 04/10 | Loss: 0.3384 | Train Acc: 0.8471
Epoch 05/10 | Loss: 0.3223 | Train Acc: 0.8542
Epoch 06/10 | Loss: 0.2991 | Train Acc: 0.8658
Epoch 07/10 | Loss: 0.2843 | Train Acc: 0.8669
Epoch 08/10 | Loss: 0.2710 | Train Acc: 0.8824
Epoch 09/10 | Loss: 0.2618 | Train Acc: 0.8778
Epoch 10/10 | Loss: 0.2461 | Train Acc: 0.8913
Training complete.


## 8. Evaluation

Final testing on the unseen test set. We prioritize F1-scores over Accuracy due to the potential class imbalance in BFRB vs. non-BFRB gestures.

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for token_ids, pad_mask, labels in loader:
            token_ids = token_ids.to(DEVICE)
            pad_mask  = pad_mask.to(DEVICE)
            logits    = model(token_ids, pad_mask)
            preds     = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


preds, labels = evaluate(model, test_loader)

binary_f1 = f1_score(labels, preds, average="binary", pos_label=1)
macro_f1  = f1_score(labels, preds, average="macro")

print(f"Binary F1-score        : {binary_f1 * 100:.2f}%")
print(f"Macro-averaged F1-score: {macro_f1  * 100:.2f}%")

d:\Documents\GitHub\2026-winter-capstone-project-group4_wecandothis\.venv\Lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Binary F1-score        : 90.46%
Macro-averaged F1-score: 85.44%


## 9. Save Results

Saves the final metrics, predictions, and hyperparameters to a file for later comparison with other models

In [ ]:
results = {
    "model"     : "Transformer",
    "binary_f1" : binary_f1,
    "macro_f1"  : macro_f1,
    "preds"     : preds,
    "labels"    : labels,
    "hyperparams": {
        "embed_dim" : EMBED_DIM,
        "num_heads" : NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "ff_dim"    : FF_DIM,
        "max_len"   : MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs"    : EPOCHS,
        "lr"        : LR
    }
}

with open(RESULTS_DIR / "transformer_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved: ../results/transformer_results.pkl")
print("Transformer Model Complete")
print(f"Binary F1  : {binary_f1 * 100:.2f}%")
print(f"Macro F1   : {macro_f1  * 100:.2f}%")

Saved: ../results/transformer_results.pkl
Transformer Model Complete
Binary F1  : 90.46%
Macro F1   : 85.44%
